# Logistic regression for concrete strength classification

In this notebook, we repeat the concrete strength classification exercise (introduced with the linear discriminant analysis notebook) with logistic regression. Rather than predicting the compressive strength from a regression perspective, we discretize the target variable into three strength classes:
- Low strength: $f_c < 25\ \text{MPa}$
- Medium strength: $25 \leq f_c \leq 40\ \text{MPa}$
- High strength: $f_c > 40\ \text{MPa}$
  
We will then reformulate the original regression problem as a multiclass classification problem and address it with logistic regression. Unlike linear discriminant analysis, logistic regression does not assume that the features follow Gaussian distributions. Instead, it directly models the posterior probability of each class by maximizing the likelihood of the observed labels.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score

### Load data

The dataset contains:
- Input variables describing the concrete composition
- One target variable corresponding to the concrete compressive strength

In [2]:
# Load data from Excel
file_path = "../Data/Concrete/Concrete.xls"
df = pd.read_excel(file_path)
# The dataset is stored in Excel format (.xls).
# Pandas requires the 'xlrd' package to read this format.
# If you get an error, install it via: pip install xlrd.

# Display the first rows of the dataset
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '../Data/Concrete/Concrete.xls'

In [ ]:
# In this dataset, the last column is the concrete compressive strength target
X = df.iloc[1:, :-1].values # All columns except last → input features
y = df.iloc[1:, -1].values # Last column → target

N, d = X.shape

print("Input shape:", X.shape)
print("Target shape:", y.shape)
print(f"Number of samples: {N}")
print(f"Number of input features: {d}")

### Convert compressive strength into strength classes

In [ ]:
# Strength intervals (MPa)
bins = [0, 25, 40, np.inf]

class_names = ["Low strength", "Medium strength", "High strength",]

y_class = pd.cut(y,bins=bins,labels=class_names)

plt.figure(figsize=(6,4))
y_class.value_counts().sort_index().plot.bar()

plt.ylabel("Number of samples")

plt.show()

### Split and normalize the data

- Split using 80% train and 20% test
- Compute normalization statistics from training data
- Consistently apply normalization to the test data

In [ ]:
# Split data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y_class, test_size=0.2, random_state=4, stratify=y_class)

# Standardization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Train the logistic regression model

Scikit-learn automatically performs multinomial logistic regression when more than two classes are present. The model estimates one linear decision function for each class and converts these scores into probabilities through the Softmax function.

In [ ]:
logreg = LogisticRegression(solver="lbfgs",max_iter=1000)

logreg.fit(X_train_scaled, y_train)

### Prediction and visualization

The performance of a classifier can be evaluated by comparing its predictions with the ground-truth labels. A convenient way to summarize the classification results is with a confusion matrix, which reports the number of correctly and incorrectly classified samples.

The confusion matrix is often complemented by the accuracy metric, which measures the proportion of correctly classified samples.

In [ ]:
y_pred = logreg.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)

cm = confusion_matrix(y_test, y_pred, normalize='true')
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Low strength", "Medium strength", "High strength"])
disp.plot(cmap='Blues', xticks_rotation=25)
disp.im_.set_clim(0, 1)
plt.title(f"Accuracy = {accuracy:.2%}")
plt.show()

### Posterior probabilities

One of the main advantages of logistic regression is that it directly estimates class probabilities. These probabilities can be interpreted as the confidence of the classifier in each prediction.

In [ ]:
probabilities = logreg.predict_proba(X_test_scaled)

pd.DataFrame(probabilities[:10], # Display only the first ten instances
             columns=class_names)

### Model coefficients

Each class is associated with a linear model $a_k(\mathbf{x})=\mathbf{x}^T\boldsymbol{\beta}_k$. Large positive entries in the parameter vector $\boldsymbol{\beta}_k$ indicate that increasing the corresponding feature increases the probability of belonging to that class. We can create an heatmap showing which variables drive each class.

In [ ]:
feature_names = ["Cement", "Blast furnace slag", "Fly ash", "Water", "Superplasticizer", "Coarse aggregate", "Fine aggregate", "Age"]
                                                                                                                                                                                                                                                                                                                                                                                                          
coef = pd.DataFrame(logreg.coef_, columns=feature_names, index=logreg.classes_)

plt.figure(figsize=(10,4))
plt.imshow(coef, aspect="auto")

plt.xticks(np.arange(len(feature_names)), feature_names, rotation=45)
plt.yticks(np.arange(len(class_names)), class_names)

plt.colorbar(label="Coefficient")
plt.title("Logistic regression coefficients")
plt.tight_layout()

### Exercises

- Compare the classification accuracy of logistic regression and LDA.
  
- Inspect the learned coefficients. Which concrete properties contribute the most to each strength class?

- Investigate how regularization affects the classification results.

- Compare the posterior probabilities produced by logistic regression with those obtained using LDA.